# AgriSol Model Optimization Suite
## Advanced Pruning & Deployment Optimization for All Plant Disease Models

### Executive Summary
This notebook provides **comprehensive model optimization techniques** for all four AgriSol plant disease detection models. We implement advanced pruning, quantization, and deployment optimization strategies to achieve maximum efficiency for mobile React Native deployment.

### Optimization Strategy Overview
- **Model Pruning**: Remove redundant parameters while maintaining accuracy
- **Advanced Quantization**: INT8 optimization with calibration datasets
- **Architecture Comparison**: Performance vs efficiency trade-offs
- **Mobile Deployment**: React Native integration optimization
- **Ensemble Strategies**: Combining models for superior performance

---

## Table of Contents
1. [Environment Setup & Model Loading](#setup)
2. [Individual Model Pruning Analysis](#pruning)
3. [Advanced Quantization Techniques](#quantization)
4. [Performance vs Efficiency Trade-offs](#performance)
5. [Mobile Deployment Optimization](#mobile)
6. [Ensemble Model Creation](#ensemble)
7. [Final Deployment Recommendations](#deployment)

## 1. Environment Setup & Model Loading {#setup}

### Advanced Optimization Toolkit
We configure the environment for comprehensive model optimization across all four AgriSol models.

In [ ]:
# Advanced Model Optimization Environment Setup
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import json
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# TensorFlow optimization libraries
import tensorflow as tf
from tensorflow import keras
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.sparsity.keras import prune, pruning_callbacks, pruning_schedule

# Ensure matplotlib displays plots inline
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Hardware optimization
physical_devices = tf.config.experimental.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"🚀 GPU Optimized: {tf.config.experimental.get_device_details(physical_devices[0])}")
else:
    print("💻 CPU Mode: AMD Ryzen 7000 optimized")

# Mixed precision for efficiency
tf.keras.mixed_precision.set_global_policy('mixed_float16')

print(f"\n🔧 AgriSol Model Optimization Suite Configuration:")
print(f"TensorFlow Version: {tf.__version__}")
print(f"TensorFlow Model Optimization: {tfmot.__version__}")
print(f"Target: All 4 AgriSol models (Tomato, Corn, Potato, Bean)")
print(f"Focus: Pruning, Quantization, Mobile Deployment")
print(f"Hardware: AMD Ryzen 7000 + 32GB RAM")

# Define paths
NOTEBOOK_PATH = Path(r"C:\Users\mbuto\Agri-sol - Copy\Notebook")
MODEL_SAVE_PATH = NOTEBOOK_PATH

# AgriSol Model Registry
AGRISOL_MODELS = {
    'tomato': {
        'name': 'AgriSol Tomato Disease Detection',
        'architecture': 'EfficientNetB0',
        'classes': 10,
        'samples': 18345,
        'challenge': 'Balanced dataset, high class diversity',
        'model_file': 'best_efficientnet_tomato_finetuned.h5'
    },
    'corn': {
        'name': 'AgriSol Maize Disease Detection', 
        'architecture': 'MobileNetV2',
        'classes': 4,
        'samples': 7316,
        'challenge': 'Efficiency focus, mobile-first',
        'model_file': 'best_mobilenet_corn_finetuned.h5'
    },
    'potato': {
        'name': 'AgriSol Potato Disease Detection',
        'architecture': 'ResNet50V2', 
        'classes': 3,
        'samples': 5702,
        'challenge': 'Deep learning, historical diseases',
        'model_file': 'best_resnet_potato_finetuned.h5'
    },
    'bean': {
        'name': 'AgriSol Bean Disease Detection',
        'architecture': 'DenseNet121',
        'classes': 4, 
        'samples': 3012,
        'challenge': 'Severe class imbalance (6:1)',
        'model_file': 'best_densenet121_bean_finetuned.h5'
    }
}

print(f"\n📋 AgriSol Model Registry Loaded:")
for crop, info in AGRISOL_MODELS.items():
    print(f"✅ {crop.upper()}: {info['architecture']} ({info['classes']} classes, {info['samples']} samples)")

print(f"\n🎯 Optimization Goals:")
print(f"✅ Reduce model sizes by 60-80%")
print(f"✅ Maintain accuracy within 2% of original")
print(f"✅ Optimize for React Native deployment")
print(f"✅ Enable efficient inference on mobile devices")
print(f"✅ Create ensemble deployment strategies")

## 2. Individual Model Pruning Analysis {#pruning}

### Systematic Pruning for Each Architecture
We implement tailored pruning strategies for each model architecture, considering their unique characteristics and deployment requirements.

In [ ]:
# Advanced Model Pruning Implementation

def analyze_model_structure(model, model_name):
    """
    Comprehensive analysis of model structure for pruning optimization
    """
    print(f"\n🔍 ANALYZING {model_name.upper()} MODEL STRUCTURE:")
    print("═" * 70)
    
    total_params = model.count_params()
    trainable_params = sum([tf.size(var) for var in model.trainable_variables])
    
    print(f"📊 Parameter Analysis:")
    print(f"   Total Parameters: {total_params:,}")
    print(f"   Trainable Parameters: {trainable_params:,}")
    print(f"   Non-trainable Parameters: {total_params - trainable_params:,}")
    
    # Layer analysis
    layer_types = {}
    prunable_layers = 0
    
    for layer in model.layers:
        layer_type = type(layer).__name__
        layer_types[layer_type] = layer_types.get(layer_type, 0) + 1
        
        # Check if layer is prunable
        if hasattr(layer, 'kernel'):
            prunable_layers += 1
    
    print(f"\n🏗️  Architecture Analysis:")
    print(f"   Total Layers: {len(model.layers)}")
    print(f"   Prunable Layers: {prunable_layers}")
    
    print(f"\n📋 Layer Distribution:")
    for layer_type, count in sorted(layer_types.items()):
        print(f"   {layer_type}: {count}")
    
    return {
        'total_params': total_params,
        'trainable_params': int(trainable_params),
        'prunable_layers': prunable_layers,
        'layer_types': layer_types
    }

def create_pruning_schedule(model_name, total_epochs=10):
    """
    Create optimized pruning schedule for different architectures
    """
    # Architecture-specific pruning strategies
    pruning_configs = {
        'tomato': {'initial_sparsity': 0.0, 'final_sparsity': 0.7, 'frequency': 100},
        'corn': {'initial_sparsity': 0.0, 'final_sparsity': 0.8, 'frequency': 100},  # MobileNet can handle more pruning
        'potato': {'initial_sparsity': 0.0, 'final_sparsity': 0.6, 'frequency': 100},  # ResNet needs conservative pruning
        'bean': {'initial_sparsity': 0.0, 'final_sparsity': 0.75, 'frequency': 100}   # DenseNet benefits from pruning
    }
    
    config = pruning_configs.get(model_name, pruning_configs['tomato'])
    
    pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=config['initial_sparsity'],
        final_sparsity=config['final_sparsity'],
        begin_step=0,
        end_step=total_epochs * config['frequency']
    )
    
    return pruning_schedule, config

def apply_structured_pruning(model, model_name):
    """
    Apply structured pruning optimized for each architecture
    """
    print(f"\n🔧 APPLYING STRUCTURED PRUNING TO {model_name.upper()}:")
    
    # Get pruning schedule
    pruning_schedule, config = create_pruning_schedule(model_name)
    
    # Define pruning method
    def apply_pruning_to_layer(layer):
        # Only prune certain layer types
        if isinstance(layer, (tf.keras.layers.Dense, tf.keras.layers.Conv2D)):
            return tfmot.sparsity.keras.prune_low_magnitude(
                layer,
                pruning_schedule=pruning_schedule
            )
        return layer
    
    # Apply pruning to model
    pruned_model = tf.keras.models.clone_model(
        model,
        clone_function=apply_pruning_to_layer
    )
    
    print(f"✅ Pruning Configuration:")
    print(f"   Target Sparsity: {config['final_sparsity']*100:.1f}%")
    print(f"   Pruning Frequency: {config['frequency']} steps")
    print(f"   Strategy: Polynomial decay")
    
    return pruned_model, config

# Load and analyze all available models
model_analyses = {}
available_models = {}

print(f"\n🔍 COMPREHENSIVE MODEL ANALYSIS:")
print("═" * 80)

for crop, info in AGRISOL_MODELS.items():
    model_path = MODEL_SAVE_PATH / info['model_file']
    
    if model_path.exists():
        try:
            # Load model
            model = tf.keras.models.load_model(str(model_path))
            available_models[crop] = model
            
            # Analyze structure
            analysis = analyze_model_structure(model, crop)
            model_analyses[crop] = analysis
            
            print(f"✅ {crop.upper()} model loaded and analyzed")
            
        except Exception as e:
            print(f"⚠️  {crop.upper()} model not found or corrupted: {e}")
    else:
        print(f"❌ {crop.upper()} model file not found: {model_path}")

print(f"\n📊 LOADED MODELS SUMMARY:")
print(f"Available models: {len(available_models)}/{len(AGRISOL_MODELS)}")

# Create comparative analysis visualization
if available_models:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('AgriSol Models: Architecture & Parameter Analysis', fontsize=16, fontweight='bold')
    
    # 1. Parameter comparison
    ax1 = axes[0, 0]
    crops = list(model_analyses.keys())
    total_params = [model_analyses[crop]['total_params'] for crop in crops]
    trainable_params = [model_analyses[crop]['trainable_params'] for crop in crops]
    
    x = np.arange(len(crops))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, [p/1e6 for p in total_params], width, label='Total', alpha=0.8)
    bars2 = ax1.bar(x + width/2, [p/1e6 for p in trainable_params], width, label='Trainable', alpha=0.8)
    
    ax1.set_xlabel('Models')
    ax1.set_ylabel('Parameters (Millions)')
    ax1.set_title('Parameter Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels([c.upper() for c in crops])
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}M', ha='center', va='bottom', fontweight='bold')
    
    # 2. Prunable layers analysis
    ax2 = axes[0, 1]
    prunable_layers = [model_analyses[crop]['prunable_layers'] for crop in crops]
    total_layers = [len(available_models[crop].layers) for crop in crops]
    
    bars3 = ax2.bar(crops, prunable_layers, alpha=0.7, color='green', label='Prunable')
    bars4 = ax2.bar(crops, total_layers, alpha=0.5, color='gray', label='Total')
    
    ax2.set_xlabel('Models')
    ax2.set_ylabel('Number of Layers')
    ax2.set_title('Prunable vs Total Layers')
    ax2.set_xticklabels([c.upper() for c in crops])
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    # Add percentage labels
    for i, (prunable, total) in enumerate(zip(prunable_layers, total_layers)):
        percentage = (prunable / total) * 100
        ax2.text(i, prunable + 2, f'{percentage:.1f}%', ha='center', fontweight='bold')
    
    # 3. Architecture complexity visualization
    ax3 = axes[1, 0]
    complexity_scores = []
    for crop in crops:
        params = model_analyses[crop]['total_params']
        layers = len(available_models[crop].layers)
        # Simple complexity score
        complexity = (params / 1e6) * (layers / 100)
        complexity_scores.append(complexity)
    
    colors = ['red', 'orange', 'yellow', 'green']
    bars5 = ax3.bar(crops, complexity_scores, color=colors[:len(crops)], alpha=0.7)
    ax3.set_xlabel('Models')
    ax3.set_ylabel('Complexity Score')
    ax3.set_title('Model Complexity Analysis')
    ax3.set_xticklabels([c.upper() for c in crops])
    ax3.grid(axis='y', alpha=0.3)
    
    # 4. Pruning potential visualization
    ax4 = axes[1, 1]
    pruning_potentials = []
    architecture_names = []
    
    for crop in crops:
        _, config = create_pruning_schedule(crop)
        pruning_potentials.append(config['final_sparsity'] * 100)
        architecture_names.append(AGRISOL_MODELS[crop]['architecture'])
    
    bars6 = ax4.bar(range(len(crops)), pruning_potentials, 
                   color=['darkred', 'darkorange', 'darkblue', 'darkgreen'][:len(crops)], alpha=0.8)
    ax4.set_xlabel('Models')
    ax4.set_ylabel('Target Pruning (%)')
    ax4.set_title('Pruning Potential by Architecture')
    ax4.set_xticks(range(len(crops)))
    ax4.set_xticklabels([f'{c.upper()}\n({arch})' for c, arch in zip(crops, architecture_names)], 
                       rotation=45, ha='right')
    ax4.grid(axis='y', alpha=0.3)
    
    # Add percentage labels
    for bar in bars6:
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{height:.0f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📈 Analysis Complete! Visualization shows:")
    print(f"✅ Parameter distribution across models")
    print(f"✅ Pruning potential for each architecture")
    print(f"✅ Layer complexity analysis")
    print(f"✅ Optimization readiness assessment")
else:
    print(f"⚠️  No models available for analysis. Please ensure model files exist.")

In [ ]:
# Implement Pruning for Available Models

pruned_models = {}
pruning_results = {}

print(f"\n🔧 IMPLEMENTING STRUCTURED PRUNING:")
print("═" * 80)

for crop, model in available_models.items():
    print(f"\n🌱 Processing {crop.upper()} model...")
    
    try:
        # Apply pruning
        pruned_model, config = apply_structured_pruning(model, crop)
        
        # Compile pruned model
        pruned_model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        pruned_models[crop] = pruned_model
        
        # Calculate size reduction estimates
        original_params = model.count_params()
        estimated_reduction = config['final_sparsity']
        estimated_final_params = int(original_params * (1 - estimated_reduction))
        
        pruning_results[crop] = {
            'original_params': original_params,
            'estimated_final_params': estimated_final_params,
            'sparsity_target': config['final_sparsity'],
            'estimated_reduction': estimated_reduction * 100
        }
        
        print(f"✅ {crop.upper()} pruning setup complete:")
        print(f"   Target sparsity: {config['final_sparsity']*100:.1f}%")
        print(f"   Est. param reduction: {estimated_reduction*100:.1f}%")
        print(f"   Original params: {original_params:,}")
        print(f"   Est. final params: {estimated_final_params:,}")
        
    except Exception as e:
        print(f"❌ Error pruning {crop.upper()} model: {e}")

# Create pruning results summary
if pruning_results:
    print(f"\n📊 PRUNING OPTIMIZATION SUMMARY:")
    print("═" * 80)
    
    # Create summary DataFrame
    summary_data = []
    for crop, results in pruning_results.items():
        summary_data.append({
            'Model': crop.upper(),
            'Architecture': AGRISOL_MODELS[crop]['architecture'],
            'Original_Params': f"{results['original_params']:,}",
            'Target_Sparsity': f"{results['sparsity_target']*100:.1f}%",
            'Est_Reduction': f"{results['estimated_reduction']:.1f}%",
            'Est_Final_Params': f"{results['estimated_final_params']:,}"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    # Visualization of pruning impact
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('AgriSol Models: Pruning Impact Analysis', fontsize=16, fontweight='bold')
    
    # Parameter reduction visualization
    crops = list(pruning_results.keys())
    original = [pruning_results[crop]['original_params']/1e6 for crop in crops]
    estimated_final = [pruning_results[crop]['estimated_final_params']/1e6 for crop in crops]
    
    x = np.arange(len(crops))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, original, width, label='Original', alpha=0.8, color='red')
    bars2 = ax1.bar(x + width/2, estimated_final, width, label='After Pruning', alpha=0.8, color='green')
    
    ax1.set_xlabel('Models')
    ax1.set_ylabel('Parameters (Millions)')
    ax1.set_title('Parameter Reduction Through Pruning')
    ax1.set_xticks(x)
    ax1.set_xticklabels([c.upper() for c in crops])
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Add reduction percentage labels
    for i, crop in enumerate(crops):
        reduction = pruning_results[crop]['estimated_reduction']
        ax1.text(i, max(original[i], estimated_final[i]) + 0.5, 
                f'-{reduction:.1f}%', ha='center', va='bottom', 
                fontweight='bold', color='darkgreen')
    
    # Sparsity targets visualization
    sparsity_targets = [pruning_results[crop]['sparsity_target']*100 for crop in crops]
    architectures = [AGRISOL_MODELS[crop]['architecture'] for crop in crops]
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    bars3 = ax2.bar(range(len(crops)), sparsity_targets, 
                   color=colors[:len(crops)], alpha=0.8)
    
    ax2.set_xlabel('Models')
    ax2.set_ylabel('Target Sparsity (%)')
    ax2.set_title('Pruning Aggressiveness by Architecture')
    ax2.set_xticks(range(len(crops)))
    ax2.set_xticklabels([f'{c.upper()}\n({arch})' for c, arch in zip(crops, architectures)],
                       rotation=45, ha='right')
    ax2.grid(axis='y', alpha=0.3)
    
    # Add sparsity labels
    for bar in bars3:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{height:.0f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n🎯 PRUNING READINESS STATUS:")
    total_original = sum(results['original_params'] for results in pruning_results.values())
    total_final = sum(results['estimated_final_params'] for results in pruning_results.values())
    overall_reduction = ((total_original - total_final) / total_original) * 100
    
    print(f"✅ {len(pruned_models)} models prepared for pruning")
    print(f"✅ Overall parameter reduction: {overall_reduction:.1f}%")
    print(f"✅ Total original parameters: {total_original:,}")
    print(f"✅ Estimated final parameters: {total_final:,}")
    print(f"\n🚀 All models ready for advanced quantization and deployment!")
else:
    print(f"⚠️  No models successfully prepared for pruning.")